[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [NumPy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)

# Aggregation


## What you will be able to do

Reduce a year of readings to the numbers people ask for with `sum`, `mean`, `min`, `max`, `std` and
`percentile`, find when the extremes happened with `argmin` and `argmax`, and get correct answers
from Svalbard's row, missing hours and all.

## The idea

### The problem

A station's year is 8,760 readings, too many to read one by one, and the questions people ask of a station's year have short answers. What
was the mean? How cold did it get, and when? How much did the readings vary, and what was a typical
hour? How much heating had the year needed by the end of March? Each answer collapses thousands of
values into one, or into a few.

In plain Python each of those is a loop that carries something along with it, a running total, the
smallest value so far, or the position of the smallest value so far. Python's own `sum`, `min` and
`max` cover the simplest cases, and they fail on Svalbard's missing hours, since `None` cannot be
added to a number or compared with one. Each loop also has to decide, separately, what to do with an
empty list.

### What an aggregation is

> An **aggregation**, also called a reduction, combines the values of an array into fewer values.
> `readings[0].mean()` turns Bergen's 8,760 readings into one number, and with `axis` a reduction
> turns each station's row into one number per station. The result has the shape of the array with
> the combined dimensions removed.

### Why it works that way

Every reduction is both a method and a function, `bergen.mean()` and `np.mean(bergen)`, and both run
in compiled code over the whole array. A reduction over a whole array returns a single NumPy number
of type `np.float64`, which prints like a Python number and shows its type when it is the last line
of a cell.

The ordinary reductions treat `nan` as a value like any other, and arithmetic with `nan` gives `nan`,
so one missing hour makes a year's mean `nan`. `min` and `max` return `nan` too, and `argmin` and
`argmax` return the position of the first `nan`. Each has a `nan` version, `np.nanmean`, `np.nanmax`,
`np.nanargmin` and the rest, which leaves the missing values out, and that is usually what a missing
reading calls for.

A reduction over no values at all has to return something or refuse. The sum of nothing is 0, the
value that adding leaves unchanged. A minimum has no such value, so NumPy raises. A mean of nothing
divides by zero, and NumPy returns `nan` with a warning.

### Where this shows up

The **Pandas, Deep Dive** guide applies these same reductions to the columns of a table and to groups
of its rows, and its `describe` prints a column's count, mean, standard deviation, minimum,
percentiles and maximum in one table. The box plots in the **Matplotlib, Deep Dive** guide draw a
median and percentiles. Scaling the columns of a dataset for a model starts from each column's mean
and standard deviation.

### What this notebook covers

- `sum` and `mean`, and the NumPy number a reduction returns
- `min`, `max`, `argmin` and `argmax`, and the dates of the extremes
- `median` and `percentile`
- `var` and `std`
- the `nan` versions, for Svalbard's row
- reductions over the whole grid, and a first look at `axis`
- running totals with `np.cumsum`

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import numpy as np

celsius = np.array([2.5, -1.0, np.nan, 4.5])
print(celsius.sum(), np.nansum(celsius))
print(np.nanmin(celsius), np.nanargmin(celsius))
print(np.nanmean(celsius))
```

```
nan 6.0
-1.0 1
2.0
```

The ordinary sum was `nan` because one value was missing. The `nan` versions left it out, found the
smallest of the other three at position 1, and took the mean of 6.0 over three values.

## Setup

Four imports, the year of readings, and a helper that finds an hour's position.

- `numpy`, imported as `np`, is the library, and the cell prints its version
- `math`, `datetime` and `timedelta` build the readings, and `hour_of_year` finds positions
- `warnings` catches the warning that one of the Common errors gives, so that its message prints
  without the path of a file inside NumPy

The last Common error imports `statistics`, Python's own module, to compare with NumPy.
`readings_list` holds four lists of 8,760 readings, one for each station in `STATIONS`, made by
`year_of_readings`, the formula the **Why NumPy** notebook explained, and `readings` is the same
values as one array of shape (4, 8760), with `nan` for Svalbard's silent day. `hour_of_year(7, 1, 12)`
gives the position of noon on 1 July along a station's row.

In [1]:
import math
import warnings
from datetime import datetime, timedelta

import numpy as np

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
START = datetime(2025, 1, 1)
HOURS = 365 * 24


def year_of_readings(station):
    """A reading for every hour of 2025 at one station, as the sqlite3 guide makes them.
    Svalbard was silent on 2 March, so those 24 hours are None."""
    i = list(STATIONS).index(station)
    row = []
    for n in range(HOURS):
        hour = START + timedelta(hours=n)
        if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
            row.append(None)
        else:
            season = -math.cos(2 * math.pi * (n - 400) / HOURS)
            day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
            wobble = ((n * 37 + i * 101) % 17 - 8) / 10
            row.append(round(STATIONS[station] + 9 * season + 3 * day + wobble, 1) + 0.0)
    return row


readings_list = [year_of_readings(station) for station in STATIONS]
readings = np.array(readings_list, dtype=float)       # None becomes nan


def hour_of_year(month, day, hour=0):
    """The position along a station's row of an hour of 2025. hour_of_year(7, 1, 12) is noon on 1 July."""
    return (datetime(2025, month, day, hour) - START) // timedelta(hours=1)


print("numpy", np.__version__, "|", len(readings_list), "stations,", len(readings_list[0]), "hours each |", readings.shape)

numpy 2.1.3 | 4 stations, 8760 hours each | (4, 8760)


## Worked examples

### sum and mean

Every reduction is a method of the array and a function in `np`, and the two give the same answer:

In [2]:
bergen = readings[0]

print(bergen.size)
print(bergen.sum().round(1))
print(bergen.mean().round(2), np.mean(bergen).round(2))
print((bergen.sum() / bergen.size).round(2))

8760
70079.0
8.0 8.0
8.0


The mean is the sum divided by the count, 70,079.0 over 8,760 hours, which is 8.0 degrees.

### A reduction returns a NumPy number

In [3]:
warmest = bergen.max()

print(type(warmest).__name__, warmest)
print(type(float(warmest)).__name__)
warmest

float64 20.8
float


np.float64(20.8)

`print` shows the number alone. A cell whose last line is the value shows `np.float64(20.8)`, the
type and the number, which is how NumPy 2 writes a NumPy number that is not printed. The **dtype**
notebook explained `float64`, and `float(warmest)` turns the value into Python's own `float` for a
library that needs one.

### The extremes and when they happened

`argmin` and `argmax` give the positions of the smallest and largest values, and a position along a
station's row is an hour of the year:

In [4]:
coldest_hour = bergen.argmin()
warmest_hour = bergen.argmax()

print(bergen.min(), bergen.max(), np.ptp(bergen))
print(coldest_hour, warmest_hour)
print(START + timedelta(hours=int(coldest_hour)))
print(START + timedelta(hours=int(warmest_hour)))

-4.8 20.8 25.6
459 4839
2025-01-20 03:00:00
2025-07-21 15:00:00


Bergen ranged from -4.8 degrees at 03:00 on 20 January to 20.8 at 15:00 on 21 July. `np.ptp`, for
peak to peak, gives the distance between the two, 25.6 degrees, in one call.

### Medians and percentiles

The median is the middle value once the readings are sorted. A percentile generalizes it, and the 10th
percentile is the value that a tenth of the readings are at or below:

In [5]:
low, middle, high = np.percentile(bergen, [10, 50, 90])

print(np.median(bergen))
print(low, middle, high)
print(round((bergen <= low).mean(), 3), round((bergen > high).mean(), 3))

8.0
-1.1 8.0 17.0
0.1 0.1


A tenth of Bergen's hours were at or below -1.1 degrees, and a tenth were above 17.0. The 50th
percentile is the median, 8.0, the same as the mean, because the formula in Setup swings as far above
the mean as below it.

### Spread with var and std

The variance is the mean squared distance from the mean, in degrees squared. The standard deviation
is its square root, back in degrees:

In [6]:
print(bergen.var().round(2))
print(bergen.std().round(2))
print(np.sqrt(bergen.var()).round(2))

45.25
6.73
6.73


A typical reading at Bergen was 6.73 degrees from the station's mean of 8.0.

### The nan versions

Svalbard's row has 24 `nan` hours. Every reduction has a `nan` version that leaves them out:

In [7]:
svalbard = readings[2]

print(svalbard.mean(), svalbard.max())
print(np.nanmean(svalbard).round(2), np.nanmin(svalbard), np.nanmax(svalbard))
print(np.nanpercentile(svalbard, [10, 50, 90]))
print(np.count_nonzero(~np.isnan(svalbard)))

nan nan
-4.48 -17.3 8.3
[-13.5  -4.5   4.5]
8736


The ordinary mean and maximum are `nan`, and the `nan` versions give Svalbard's mean as -4.48 over
the 8,736 hours it reported.

### The whole grid, and a first look at axis

Without `axis`, a reduction combines every value in the grid into one. With `axis=1` it combines
along the hours and keeps one value per station:

In [8]:
print(readings.size, np.nanmax(readings))
print(np.nanmax(readings, axis=1))
print(np.nanmax(readings, axis=1).shape)

35040 20.8
[20.8 19.3  8.3 16.2]
(4,)


The warmest of all 35,040 readings was Bergen's 20.8, and each station's own warmest came back as an
array of shape (4,), one value for each row. **The axis Argument** notebook explains which number to
give `axis` for which question.

### Running totals with np.cumsum

`np.cumsum` keeps every partial sum instead of only the last. Applied to Bergen's daily heating
degree days from the **Vectorized Math** notebook, it gives the heating the year had needed by the
end of each day:

In [9]:
bergen_daily = bergen.reshape(365, 24).mean(axis=1)
heating = np.maximum(15.5 - bergen_daily, 0.0)
running = np.cumsum(heating)

print(running.shape)
print(running[:3].round(1), running[-1].round(1))

halfway = np.argmax(running >= running[-1] / 2)
print(halfway, (START + timedelta(days=int(halfway))).date())

(365,)
[16.2 32.4 48.6] 2805.3
98 2025-04-09


`running` ends at 2,805.3, the yearly total. The comparison `running >= running[-1] / 2` is `False`
up to the halfway day and `True` from then on. `np.argmax` gives the first position of the largest
value, and in a mask that is the first `True`, so Bergen had needed half its year's heating by
9 April.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/10-aggregation-solutions.ipynb).

**1.** Print Oslo's mean, row 1, rounded to two places, and the date and time of its warmest hour.

In [10]:
# your code here

**2.** Print the 25th, 50th and 75th percentiles of Tromso's readings, row 3.

In [11]:
# your code here

**3.** Print Svalbard's standard deviation with `np.nanstd`, rounded to two places, and the date and time of its warmest hour with `np.nanargmax`.

In [12]:
# your code here

**4.** Find the coldest reading in the whole grid with `np.nanmin`. `np.nanargmin` on the grid gives a position in the flattened array, and `np.unravel_index(position, readings.shape)` turns it into a row and a column; print both, and the station's name.

In [13]:
# your code here

**5.** Make Oslo's running count of hours below freezing with `np.cumsum` on a mask, and print the date and time of its 1,000th frost hour.

In [14]:
# your code here

## Common errors

### No error, and a mean of nan

In [15]:
print(readings.mean())
print(readings.mean(axis=1).round(2))

nan
[8.  6.5 nan 3.5]


One missing hour made the mean of the whole grid `nan`, and the station means show which row it came
from. A mean is arithmetic over every value, and any arithmetic with `nan` gives `nan`. Use
`np.nanmean`, or decide what the missing hours should be and fill them first, as the **Indexing and
Slicing** notebook did with Svalbard's silent day.

### RuntimeWarning: Mean of empty slice

Daily means with `np.nanmean`, to leave out the missing hours, meet a day on which every hour is
missing:

In [16]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    daily_means = np.nanmean(readings.reshape(4, 365, 24), axis=2)

print(f"{caught[0].category.__name__}: {caught[0].message}")
print(np.isnan(daily_means).sum(), np.flatnonzero(np.isnan(daily_means[2])))

1 [60]


Svalbard's 2 March, day 60 counted from 0, has no readings left once the missing ones are left out,
so its mean is a mean of nothing, and `np.nanmean` gives `nan` with a warning. A day with no readings
should have no mean, so the result needs no fix, only the check in the last line that the empty
slices are the ones you expected. The `warnings` lines catch the warning so that the cell can print
it, and in your own code it appears under the cell.

### ValueError: zero-size array to reduction operation minimum which has no identity

In [17]:
svalbard_warm = svalbard[svalbard > 10]
print(svalbard_warm.size)

svalbard_warm.min()

0


ValueError: zero-size array to reduction operation minimum which has no identity

Svalbard was never above 10 degrees, so the selection is empty, and the minimum of no values has no
answer. A sum has one, 0, because adding 0 leaves any number unchanged, and that value is what the
message calls an identity. Check the size before reducing, or give `min` a value to start from with
`initial`:

In [18]:
print(svalbard_warm.sum())
print(svalbard_warm.min(initial=np.inf))

0.0
inf


### No error, and the coldest hour was a missing one

In [19]:
position = np.argmin(svalbard)

print(position, svalbard[position])
print(START + timedelta(hours=int(position)))

1440 nan
2025-03-02 00:00:00


`np.argmin` returns the position of the first `nan` whenever there is one, to match `min`, which
returns `nan`. So the position of Svalbard's coldest hour came back as its first missing hour,
midnight on 2 March. `np.nanargmin` leaves the missing hours out:

In [20]:
position = np.nanargmin(svalbard)

print(position, svalbard[position])
print(START + timedelta(hours=int(position)))

267 -17.3
2025-01-12 03:00:00


### No error, and a standard deviation that differs from statistics.stdev

In [21]:
import statistics

july_1 = bergen[hour_of_year(7, 1):hour_of_year(7, 2)]

print(july_1.size)
print(round(np.std(july_1), 3), round(statistics.stdev(july_1.tolist()), 3))

24
2.153 2.199


`np.std` divides by the number of values, 24, and gives the spread of these 24 hours on their own.
`statistics.stdev` divides by one fewer, 23, which is the estimate to use when the hours are a sample
of a larger set. `ddof`, for delta degrees of freedom, is the number `np.std` takes from the count, so
`ddof=1` gives `statistics.stdev`'s answer, and `statistics.pstdev` gives NumPy's default:

In [22]:
print(round(np.std(july_1, ddof=1), 3), round(statistics.pstdev(july_1.tolist()), 3))
print(np.std(bergen).round(4), np.std(bergen, ddof=1).round(4))

2.199 2.153
6.7271 6.7274


Over a day of 24 hours the two differ in the second decimal place, and over the year's 8,760 only in
the fourth. Say which one a number is whenever you compare it with another program's.

## Recap

- Reduce an array with `sum`, `mean`, `min`, `max`, `var` and `std`, as methods or as `np` functions
- Find when an extreme happened with `argmin` and `argmax`, and turn the position into a date
- Describe a distribution with `np.median` and `np.percentile`
- Use the `nan` versions on rows with missing values, and check for empty selections before reducing
- Keep every partial sum with `np.cumsum`, and choose `ddof` to match the standard deviation you mean

## What is next

**The axis Argument** notebook takes every reduction in this notebook along one dimension at a time,
`axis=0`, `axis=1` or a tuple of both, and shows how to keep the result's shape for broadcasting.

---

&#8592; **Previous:** [Vectorized Math](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/09-vectorized-math.ipynb)  &nbsp;·&nbsp;  [NumPy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)
